In [ ]:
import numpy as np
import rioxarray as rx
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path
import leafmap
from overrides.typing_utils import unknown
from rasterio.crs import CRS
from sgp4.earth_gravity import wgs84
import xarray as xr
from satpy.enhancements.enhancer import get_enhanced_image
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.decomposition import PCA
import joblib
import matplotlib.colors as mcolors

Data Select and filter to 650 x 512 pixels

In [ ]:
data_dir = Path('~/Downloads/sars_data').expanduser()
ds_path = sorted(data_dir.rglob('*VIIRS_corrected*2024*.nc*'))
plot_dir = data_dir / 'p2_plots'

In [ ]:
ds_path

In [ ]:
ds = xr.open_dataset(ds_path[0])

In [ ]:
ds = ds.isel(y=slice(2935-256, 2935+256), x=slice(1448-320, 1448+320))

In [ ]:
ds = ds.isel(y=slice(None, None, -1), x=slice(None, None, -1))

In [ ]:
ds

In [ ]:
img = get_enhanced_image(ds['true_color'])
img = img.data.transpose('y','x','bands').values
# img = img.data.values

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from satpy.writers import get_enhanced_image

# Get enhanced image and reshape to (y, x, bands)
img = get_enhanced_image(ds['true_color'])
rgb = img.data.transpose('y', 'x', 'bands').values

# Clip to [0, 1] for imshow
rgb = np.clip(rgb, 0, 1)

# Extract lat/lon arrays
lon = ds['longitude'].values  # shape (y, x)
lat = ds['latitude'].values   # shape (y, x)

# Build tick positions in pixel space mapped to lon/lat values
fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
ax.imshow(rgb, origin='upper')

# --- X axis: longitude ---
n_xticks = 6
x_pixel_pos = np.linspace(0, rgb.shape[1] - 1, n_xticks, dtype=int)
x_lon_vals   = lon[rgb.shape[0] // 2, x_pixel_pos]   # sample along middle row
ax.set_xticks(x_pixel_pos)
ax.set_xticklabels([f"{v:.1f}°E" for v in x_lon_vals], fontsize=9)

# --- Y axis: latitude ---
n_yticks = 6
y_pixel_pos = np.linspace(0, rgb.shape[0] - 1, n_yticks, dtype=int)
y_lat_vals   = lat[y_pixel_pos, rgb.shape[1] // 2]   # sample along middle column
ax.set_yticks(y_pixel_pos)
ax.set_yticklabels([f"{v:.1f}°N" for v in y_lat_vals], fontsize=9)

ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title("VIIRS True Color", fontsize=13, pad=10)

plt.tight_layout()
plt.savefig("/Users/cwelch/Downloads/p2_true_color_frame.png", bbox_inches='tight')
plt.show()

K-means

In [ ]:
ds

In [ ]:
BANDS = ['M01', 'M02', 'M03', 'M04', 'M05', 'M06',
         'M07', 'M08', 'M09', 'M10', 'M11', 'M12',
         'M13', 'M14', 'M15', 'M16',]

X = np.stack([ds[b].values for b in BANDS], axis=-1)
ny, nx, nb = X.shape
X = X.reshape(-1, nb)

mask = np.any(np.isnan(X), axis=1)
X_valid = X[~mask]

X_scaled = StandardScaler().fit_transform(X_valid)

In [ ]:
range_n_clusters = np.arange(2, 12)

for n_clusters in range_n_clusters:
    # Create a subplot
    fig, ax1 = plt.subplots(1, 1)
    ax1.set_xlim([-0.1, 1])
    ax1.set_ylim([0, len(X_scaled) + (n_clusters + 1) * 10])

    # Initialize clusterer
    clusterer = KMeans(n_clusters=n_clusters, random_state=10)
    cluster_labels = clusterer.fit_predict(X_scaled)

    # Calculate the average silhouette score
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    print(f"For n_clusters = {n_clusters}, average silhouette_score: {silhouette_avg:.3f}")

    # Compute silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(X_scaled, cluster_labels)

    y_lower = 10
    for i in range(n_clusters):
        # Aggregate the silhouette scores for samples belonging to cluster i and sort them
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = plt.cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_values,
                          facecolor=color, edgecolor=color, alpha=0.7)

        # Label the silhouette plots with their cluster numbers at the middle
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10

    ax1.set_title(f"Silhouette plot for {n_clusters} clusters")
    ax1.set_xlabel("Silhouette coefficient values")
    ax1.set_ylabel("Cluster label")

    # Draw the vertical line for average silhouette score
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")
    plt.show()

In [ ]:
results = {}

for k in range(2, 13):
    model = KMeans(n_clusters=k, random_state=670).fit(X_scaled)
    results[k] = {
        'model': model,
        'labels': model.labels_,
        'inertia': model.inertia_
    }

# Now you can query any k without re-running:
# cloud_clusters = results[5]['labels'].reshape(height, width)

In [ ]:
joblib.dump(results, "viirs_all_clusters_2_to_12.pkl")

In [ ]:
loaded_models = joblib.load('viirs_all_clusters_2_to_12.pkl')

In [ ]:
for model in loaded_models.values():

    k = model['model'].n_clusters

    cluster_map = model['labels'].reshape(512, 640)
    # 1. Identify the number of clusters in your CURRENT labels
    unique_labels = np.unique(model['labels'])
    n_clusters = k


    cmap_base = plt.get_cmap('tab20')
    colors = [cmap_base(i) for i in range(n_clusters)]

    custom_cmap = mcolors.ListedColormap(colors)

    fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
    # Using boundary norm ensures each integer maps to one color block perfectly
    norm = mcolors.BoundaryNorm(np.arange(n_clusters + 1) - 0.5, n_clusters)

    im = ax.imshow(cluster_map, cmap=custom_cmap, norm=norm)


    ax.set_title(f'VIIRS Cluster Classification (clusters = {k})')
    # 4. Colorbar with exact ticks
    # Setting boundaries and values makes the colorbar "step" correctly
    cbar = plt.colorbar(im, ticks=np.arange(n_clusters), fraction=0.046, pad=0.04)
    cbar.set_label('Cluster ID')
    cbar.ax.set_yticklabels([f'Cluster {i}' for i in range(1, n_clusters+1)])

    # --- X axis: longitude ---
    n_xticks = 6
    x_pixel_pos = np.linspace(0, rgb.shape[1] - 1, n_xticks, dtype=int)
    x_lon_vals   = lon[rgb.shape[0] // 2, x_pixel_pos]   # sample along middle row
    ax.set_xticks(x_pixel_pos)
    ax.set_xticklabels([f"{v:.1f}°E" for v in x_lon_vals], fontsize=9)

    # --- Y axis: latitude ---
    n_yticks = 6
    y_pixel_pos = np.linspace(0, rgb.shape[0] - 1, n_yticks, dtype=int)
    y_lat_vals   = lat[y_pixel_pos, rgb.shape[1] // 2]   # sample along middle column
    ax.set_yticks(y_pixel_pos)
    ax.set_yticklabels([f"{v:.1f}°N" for v in y_lat_vals], fontsize=9)

    fig.savefig(f'cluster_map_kmeans{k}.png')
    plt.show()

Principal Component Analysis

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# 3. Analyze Explained Variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# 4. Visualization: The Scree Plot
plt.figure(figsize=(8, 4))
plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7, label='Individual')
plt.step(range(1, len(cumulative_variance) + 1), cumulative_variance, where='mid', label='Cumulative')
plt.ylabel('Explained Variance Ratio')
plt.xlabel('Principal Component Index')
plt.title('VIIRS PCA Variance Analysis')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# 1. Select the component you want to visualize (PC1 is index 0)
pc_to_plot = X_pca[:, 0].reshape(512, 640)

# 2. Set your desired number of levels (e.g., 2, 6, or 11)
n_levels = 6

# 3. Define the Color Logic
if n_levels == 2:
    colors = ['black', 'white']
else:
    # Use tab10 for consistent colors across different runs
    cmap_base = plt.get_cmap('tab10')
    colors = [cmap_base(i) for i in range(n_levels)]

custom_cmap = mcolors.ListedColormap(colors)

# 4. Create the Boundaries
# This divides the min/max of the PCA data into equal steps
bounds = np.linspace(pc_to_plot.min(), pc_to_plot.max(), n_levels + 1)
norm = mcolors.BoundaryNorm(bounds, n_levels)

# 5. Plotting
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(pc_to_plot, cmap=custom_cmap, norm=norm, interpolation='nearest')

# 6. Discrete Colorbar
# We place the ticks in the middle of each bound segment
tick_locs = (bounds[:-1] + bounds[1:]) / 2
cbar = plt.colorbar(im, ticks=tick_locs, fraction=0.046, pad=0.04)
cbar.set_label('PC1 Variance Levels')

# Label the ticks with the actual PCA value ranges
cbar.ax.set_yticklabels([f'Level {i}' for i in range(n_levels)])

plt.title(f"VIIRS PC1 Discretized into {n_levels} Levels")
plt.axis('off')
plt.show()

In [ ]:
X_pca[:]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

height = 512
width = 640

def plot_pca_discrete(pca_data, height, width, n_levels=6, title="PCA Component", rgb=rgb):
    """
    Plots a 2D PCA component discretized into a specific number of levels
    with a perfectly aligned discrete colorbar.
    """
    # 1. Reshape the 1D PCA scores back to the 2D image frame
    pc_image = pca_data.reshape(height, width)

    # Using tab10 (up to 10) or tab20 (up to 20) for consistent indexing
    cmap_base = plt.get_cmap('tab10' if n_levels <= 10 else 'tab20')
    colors = [cmap_base(i) for i in range(n_levels)]

    custom_cmap = mcolors.ListedColormap(colors)

    # 3. Create boundaries based on the data range
    # This splits the PC values into exactly n_levels buckets
    bounds = np.linspace(np.nanmin(pc_image), np.nanmax(pc_image), n_levels + 1)
    norm = mcolors.BoundaryNorm(bounds, n_levels)

    # 4. Create Plot
    fig, ax = plt.subplots(figsize=(10, 8))

    # 'nearest' interpolation prevents blurring between the discrete levels
    im = ax.imshow(pc_image, cmap=custom_cmap, norm=norm, interpolation='nearest')

    # 5. Configure Discrete Colorbar
    # Place ticks at the midpoint of each boundary level
    tick_locs = (bounds[:-1] + bounds[1:]) / 2
    cbar = plt.colorbar(im, ticks=tick_locs, fraction=0.046, pad=0.04)
    cbar.set_label('PCA Variance Level')

    # Optional: Label with Level index
    cbar.ax.set_yticklabels([f'Level {i}' for i in range(n_levels)])
    # --- X axis: longitude ---
    n_xticks = 6
    x_pixel_pos = np.linspace(0, rgb.shape[1] - 1, n_xticks, dtype=int)
    x_lon_vals   = lon[rgb.shape[0] // 2, x_pixel_pos]   # sample along middle row
    ax.set_xticks(x_pixel_pos)
    ax.set_xticklabels([f"{v:.1f}°E" for v in x_lon_vals], fontsize=9)

    # --- Y axis: latitude ---
    n_yticks = 6
    y_pixel_pos = np.linspace(0, rgb.shape[0] - 1, n_yticks, dtype=int)
    y_lat_vals   = lat[y_pixel_pos, rgb.shape[1] // 2]   # sample along middle column
    ax.set_yticks(y_pixel_pos)
    ax.set_yticklabels([f"{v:.1f}°N" for v in y_lat_vals], fontsize=9)

    ax.set_title(f"{title} (n_levels = {n_levels})")
    ax.axis('off')

    plt.tight_layout()
    plt.show()

# --- Example Usage ---
# To plot the 1st component with 2 colors (Black/White)
plot_pca_discrete(X_pca[:, 0], height, width, n_levels=2, title="PC1: Albedo/Brightness")

# To plot the 2nd component with 11 colors
plot_pca_discrete(X_pca[:, 1], height, width, n_levels=2, title="PC2: Thermal Contrast")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_pca_rgb(pca_scores, height, width, pc_indices=(0, 1, 2), stretch=2):
    """
    Creates an RGB composite from three Principal Components.
    pc_indices: tuple of which PCs to map to (R, G, B). Default is (PC1, PC2, PC3).
    stretch: percentile to clip from both ends (e.g., 2nd and 98th percentiles).
    """
    # 1. Extract the three chosen components
    r_pc = pca_scores[:, pc_indices[0]].reshape(height, width)
    g_pc = pca_scores[:, pc_indices[1]].reshape(height, width)
    b_pc = pca_scores[:, pc_indices[2]].reshape(height, width)

    rgb_stack = np.stack([r_pc, g_pc, b_pc], axis=2)

    # 2. Scale each channel to [0, 1] using Percentile Stretching
    def scale_channel(ch):
        vmin, vmax = np.nanpercentile(ch, [stretch, 100 - stretch])
        ch_scaled = (ch - vmin) / (vmax - vmin)
        return np.clip(ch_scaled, 0, 1)

    rgb_image = np.zeros((height, width, 3))
    for i in range(3):
        rgb_image[:, :, i] = scale_channel(rgb_stack[:, :, i])

    # 3. Plotting
    plt.figure(figsize=(12, 10))
    plt.imshow(rgb_image, interpolation='lanczos')

    plt.title(f"PCA RGB Composite (R:PC1, G:PC2, B:PC0)")
    plt.axis('off')
    plt.show()

# --- Example Usage ---
# Ensure your X_pca has at least 3 components
plot_pca_rgb(X_pca, height, width, pc_indices=(1, 2, 0), stretch=2)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Assume 'band_names' = ['M05_Vis', 'M07_NIR', 'M12_MWIR', 'M15_TIR', ...]
# pca.components_ has shape (n_components, n_features)

def plot_pca_loadings(pca_model, band_names):
    loadings = pd.DataFrame(
        pca_model.components_.T,
        columns=[f'PC{i+1}' for i in range(len(band_names))],
        index=band_names
    )

    # Plotting the first 3 or 4 components
    loadings.iloc[:, :4].plot(kind='bar', figsize=(12, 6))
    plt.axhline(0, color='black', linestyle='--', linewidth=1)
    plt.title("PCA Loadings: Contribution of each VIIRS Band to PCs")
    plt.ylabel("Weight (Loading Score)")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
band_names = ['M01', 'M02', 'M03', 'M04', 'M05', 'M06', 'M07', 'M08', 'M09', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16']

plot_pca_loadings(pca, band_names)


In [ ]:
X_input = X_pca[:, :2]

results_kmeans_pca = {}

for k in range(2, 13):
    model = KMeans(n_clusters=k, random_state=670).fit(X_input)
    results_kmeans_pca[k] = {
        'model': model,
        'labels': model.labels_,
        'inertia': model.inertia_
    }
joblib.dump(results, "viirs_kmeans_pca_clusters_2_to_12.pkl")
loaded_models_kmeans_pca = joblib.load('viirs_kmeans_pca_clusters_2_to_12.pkl')


In [ ]:
for model in loaded_models.values():

    k = model['model'].n_clusters

    cluster_map = model['labels'].reshape(height, width)
    # 1. Identify the number of clusters in your CURRENT labels
    unique_labels = np.unique(model['labels'])
    n_clusters = k


    cmap_base = plt.get_cmap('tab20')
    colors = [cmap_base(i) for i in range(n_clusters)]

    custom_cmap = mcolors.ListedColormap(colors)

    fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
    # Using boundary norm ensures each integer maps to one color block perfectly
    norm = mcolors.BoundaryNorm(np.arange(n_clusters + 1) - 0.5, n_clusters)

    im = ax.imshow(cluster_map, cmap=custom_cmap, norm=norm)


    ax.set_title(f'VIIRS Cluster Classification (clusters = {k})')
    # 4. Colorbar with exact ticks
    # Setting boundaries and values makes the colorbar "step" correctly
    cbar = plt.colorbar(im, ticks=np.arange(n_clusters), fraction=0.046, pad=0.04)
    cbar.set_label('Cluster ID')
    cbar.ax.set_yticklabels([f'Cluster {i}' for i in range(1, n_clusters+1)])

    # --- X axis: longitude ---
    n_xticks = 6
    x_pixel_pos = np.linspace(0, rgb.shape[1] - 1, n_xticks, dtype=int)
    x_lon_vals   = lon[rgb.shape[0] // 2, x_pixel_pos]   # sample along middle row
    ax.set_xticks(x_pixel_pos)
    ax.set_xticklabels([f"{v:.1f}°E" for v in x_lon_vals], fontsize=9)

    # --- Y axis: latitude ---
    n_yticks = 6
    y_pixel_pos = np.linspace(0, rgb.shape[0] - 1, n_yticks, dtype=int)
    y_lat_vals   = lat[y_pixel_pos, rgb.shape[1] // 2]   # sample along middle column
    ax.set_yticks(y_pixel_pos)
    ax.set_yticklabels([f"{v:.1f}°N" for v in y_lat_vals], fontsize=9)

    fig.savefig(f'cluster_map_kmeans{k}_pca.png')
    plt.show()